In [ ]:
import numpy as np

def elastic_1d(m1, m2, u1, u2):
    """Return post-collision velocities for a 1D elastic collision."""
    v1 = ((m1 - m2) * u1 + 2 * m2 * u2) / (m1 + m2)
    v2 = ((m2 - m1) * u2 + 2 * m1 * u1) / (m1 + m2)
    return v1, v2

# Classic check: equal masses exchange velocities
print(elastic_1d(1.0, 1.0, 3.0, 0.0))   # -> (0.0, 3.0)

In [ ]:
def collide_1d(m1, m2, u1, u2, e=1.0):
    """1D collision with restitution e. e=1 elastic, e=0 perfectly inelastic."""
    total_m = m1 + m2
    # center-of-mass velocity is unchanged by the collision
    v_cm = (m1 * u1 + m2 * u2) / total_m
    v1 = v_cm - e * (m2 / total_m) * (u1 - u2)
    v2 = v_cm + e * (m1 / total_m) * (u1 - u2)
    return v1, v2

In [ ]:
def p_total(m1, m2, v1, v2):
    return m1*v1 + m2*v2

def ke_total(m1, m2, v1, v2):
    return 0.5*m1*v1**2 + 0.5*m2*v2**2

In [ ]:

# Scenario
m1, m2 = 2.0, 3.0
u1, u2 = 4.0, 0.0

# Try multiple coefficients of restitution
e_values = [1.0, 0.5, 0.0]

print(f"{'e':>4} | {'v1 (m/s)':>10} | {'v2 (m/s)':>10} | {'p (kg·m/s)':>12} | {'KE (J)':>10}")
print("-" * 60)

for e in e_values:
    v1, v2 = collide_1d(m1, m2, u1, u2, e=e)
    p = p_total(m1, m2, v1, v2)
    ke = ke_total(m1, m2, v1, v2)
    print(f"{e:>4.1f} | {v1:>10.4f} | {v2:>10.4f} | {p:>12.4f} | {ke:>10.4f}")

In [ ]:
# ---- Initial values: the three limiting cases, all with e = 1 ----
cases = [
    ("equal masses",   1.0,   1.0, 3.0, 0.0),
    ("light -> heavy", 0.1, 100.0, 5.0, 0.0),
    ("heavy -> light", 100.0, 0.1, 5.0, 0.0),
]

In [ ]:
print(f"{'case':<16s}{'v1':>10s}{'v2':>10s}{'dp':>14s}{'dKE':>14s}")
for label, m1, m2, u1, u2 in cases:
    v1, v2 = collide_1d(m1, m2, u1, u2, e=1.0)
    dp  = p_total(m1, m2, v1, v2)  - p_total(m1, m2, u1, u2)
    dke = ke_total(m1, m2, v1, v2) - ke_total(m1, m2, u1, u2)
    print(f"{label:<16s}{v1:10.5f}{v2:10.5f}{dp:14.2e}{dke:14.2e}")

In [ ]:
def collide_2d(m1, m2, r1, r2, v1, v2, e=1.0):
    """Resolve a 2D collision between two circles. Positions r, velocities v are 2-vectors."""
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    v1, v2 = np.asarray(v1, float), np.asarray(v2, float)

    n = r2 - r1
    dist = np.linalg.norm(n)
    if dist == 0:
        return v1, v2            # degenerate; skip
    n = n / dist                 # unit normal

    # normal (scalar) components along n
    u1n, u2n = np.dot(v1, n), np.dot(v2, n)
    # tangential vector components (unchanged)
    v1t = v1 - u1n * n
    v2t = v2 - u2n * n

    # 1D collision on the normal components
    new1n, new2n = collide_1d(m1, m2, u1n, u2n, e)

    return v1t + new1n * n, v2t + new2n * n

In [ ]:
def overlapping(r1, r2, radius1, radius2):
    return np.linalg.norm(np.asarray(r2) - np.asarray(r1)) < (radius1 + radius2)

In [ ]:
# ---- Initial values: equal masses, glancing (offset) impact ----
m1, m2 = 1.0, 1.0
r1, r2 = [0.0, 0.0], [1.0, 0.5]     # offset in y -> glancing, not head-on
v1, v2 = [3.0, 0.0], [0.0, 0.0]     # body 1 moving +x, body 2 at rest
R1, R2 = 0.1, 0.1

In [ ]:
# Only resolve if the bodies are actually moving toward each other
approaching = np.dot(np.asarray(v2) - np.asarray(v1), np.asarray(r2) - np.asarray(r1)) < 0
if overlapping(r1, r2, R1, R2) and approaching:
    v1, v2 = collide_2d(m1, m2, r1, r2, v1, v2, e)